# Lab 5: Heterogeneous Price Elasticities

> Requires the artifact written by **Lab 2**.

Lab 4 produced a single number: on average a 1% price rise costs roughly 1.4% of units
sold. A seller does not set prices for the average product, though, so the more useful
question is which products are sensitive to price and which are not.

This notebook estimates the elasticity as a function of product characteristics.

## From ACE to CACE

Recall $A_{it} = a_t(S_{it}, \epsilon_{it})$, the price sensitivity specific to a product
and period. Lab 4 estimated its unconditional mean. Here we estimate the conditional
average causal effect,

$$\alpha_t(S_{it}) = E[A_{it}\mid S_{it}],$$

the part of the variation in price sensitivity that is predictable from what we observe.

Proposition 1 of the paper shows that, under the same structural assumptions as Lab 4, the
CACE is identified by the conditional average predictive effect,

$$\alpha_t(S_{it}) = \delta_t(S_{it}) :=
\frac{E\!\left[Q^{\perp}_{it}P^{\perp}_{it}\mid S_{it}\right]}
     {E\!\left[(P^{\perp}_{it})^{2}\mid S_{it}\right]}.$$

That is, partial out the state as in Lab 4, then let the slope of the
residual-on-residual regression vary with the characteristics of interest.

## The specification

Following the paper's Model II, the elasticity is linear in a small basis:

$$\alpha(S_{it}) = a_0 + \sum_{k=1}^{5}\alpha_k X^{sim}_{i,k}
                 + b_1 P_{i,t-1} + b_2 Q_{i,t-1}.$$

The two kinds of modifier answer different questions. The cluster similarities
$X^{sim}_{i,k}$ describe what kind of product it is; Lab 3 showed that five similarities
carry nearly as much information as the full 256-dimensional embedding, which is what
keeps the model interpretable. Lagged price and quantity describe how expensive and how
popular it currently is.

The intercept $a_0$ is the average effect and should land near Lab 4's estimate.

We estimate this three ways, the paper's Models II-1, II-2 and II-3, again on the
estimation products only and with standard errors clustered by product.

## Setup

In [ ]:
%pip install -q doubleml lightgbm

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from doubleml import DoubleMLData, DoubleMLPLR
from lightgbm import LGBMRegressor
from scipy.linalg import sqrtm
from scipy.stats import chi2, norm

palette = sns.color_palette("colorblind")
warnings.filterwarnings("ignore", message="X does not have valid feature names")
pd.set_option("display.width", 140)

CONFIDENCE = 0.90
RANK_TO_DEMAND = 2.0
SEED = 42

## Loading the artifact from Lab 2

In [ ]:
ARTIFACT = "subsample_v2.parquet"
drive_path = f"/content/drive/MyDrive/demand_labs_v2/{ARTIFACT}"

# Colab gives every notebook its own machine, so /content does not carry across
# labs. Google Drive does.
if os.path.isdir("/content") and not os.path.exists(drive_path):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"could not mount Drive ({type(exc).__name__})")

path = next((p for p in [drive_path, f"/content/{ARTIFACT}", ARTIFACT]
             if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError(
        "subsample_v2.parquet not found. Run Lab 2 "
        "(02_finetune_embeddings.ipynb) first and let it save to Google Drive, "
        "or upload the file into this session.")

df = pd.read_parquet(path)
print(f"loaded {path}")
print(f"{df['ASIN'].nunique():,} products, {len(df):,} rows, "
      f"{df['period'].nunique()} periods")
print(df.groupby("split").agg(products=("ASIN", "nunique"), rows=("ASIN", "size")))

In [ ]:
emb_cols = [c for c in df.columns if c.startswith("emb_")]
pca_cols = [c for c in df.columns if c.startswith("pca_")]
sim_cols = [c for c in df.columns if c.startswith("similarity_cluster_")]
sub_cols = [c for c in df.columns if c.startswith("sub_")]

lag_controls = ["Q_t-1", "P_bb_t-1"]
tab_controls = ["RATING_t-1", "REVIEW_COUNT_t-1", "n_offers",
                "n_offers_fba", "n_offers_fbm", "lightning_deal", "is_fba"]

periods = sorted(df["period"].unique())[1:]   # first period is the baseline
period_cols = [f"period_{p}" for p in periods]
for p in periods:
    df[f"period_{p}"] = (df["period"] == p).astype(int)

OUTCOME, TREATMENT = "Q_t", "P_bb_t"
X_o = tab_controls + sub_cols + period_cols

data = (df[df["split"] == "estimation"]
        .dropna(subset=lag_controls + tab_controls + [OUTCOME, TREATMENT])
        .reset_index(drop=True))
clusters = data["ASIN"]

print(f"estimation sample: {len(data):,} observations, "
      f"{data['ASIN'].nunique()} products")

## The basis

Seven modifiers plus an intercept. The lagged variables are centred and standardised, so
their coefficients read as the change in elasticity per standard deviation of popularity
or price. The similarities are centred only, since they already share a common cosine
scale.

In [ ]:
def make_basis(frame, scaled, unscaled):
    """Intercept + standardised `scaled` columns + centred `unscaled` columns."""
    out = pd.DataFrame({"intercept": np.ones(len(frame))}, index=frame.index)
    for c in scaled:
        out[c] = (frame[c] - frame[c].mean()) / frame[c].std()
    for c in unscaled:
        out[c] = frame[c] - frame[c].mean()
    return out


basis = make_basis(data, scaled=lag_controls, unscaled=sim_cols)
BASIS_COLS = list(basis.columns)
SIM_IDX = [BASIS_COLS.index(c) for c in sim_cols]
MOD_IDX = [i for i, c in enumerate(BASIS_COLS) if c != "intercept"]

print("basis:", BASIS_COLS)
basis.head(3).round(3)

## Three estimators

Models II-1 and II-2 are linear. We residualise $Q$ and $P$ on the controls by OLS, then
regress the outcome residual on the basis interacted with the price residual. Model II-2
adds interactions between the lagged state and the similarities to the control function.

Model II-3 is partially linear. The second stage is the same, but the residuals come from
cross-fitted boosted trees via `DoubleML`, so the control function is nonparametric.

In [ ]:
def lin_model_cate(frame, basis, controls, treatment=TREATMENT, outcome=OUTCOME):
    """Residual-on-residual regression with a varying slope (Models II-1, II-2)."""
    X = sm.add_constant(frame[controls], has_constant="add")
    y_tilde = frame[outcome] - sm.OLS(frame[outcome], X).fit().predict(X)
    d_tilde = frame[treatment] - sm.OLS(frame[treatment], X).fit().predict(X)
    interacted = basis.mul(d_tilde.values, axis=0)
    return sm.OLS(y_tilde, interacted).fit(
        cov_type="cluster", cov_kwds={"groups": clusters})


# Model II-2 needs the interaction terms in the *control* function
inter_cols = []
for lag in lag_controls:
    for s in sim_cols:
        name = f"int_{lag}_{s}"
        data[name] = data[lag] * data[s]
        inter_cols.append(name)

controls_base = lag_controls + emb_cols + X_o

models = {}
models["II-1 Linear"] = lin_model_cate(data, basis, controls_base)
models["II-2 Linear (interactions)"] = lin_model_cate(
    data, basis, controls_base + inter_cols)
print("linear specifications fitted")

In [ ]:
dml_controls = lag_controls + sim_cols + X_o
dml_data = DoubleMLData(data[[OUTCOME, TREATMENT, "ASIN"] + dml_controls],
                        y_col=OUTCOME, d_cols=TREATMENT,
                        x_cols=dml_controls, cluster_cols="ASIN")

learner = dict(n_estimators=500, learning_rate=0.02, random_state=SEED, verbose=-1)
np.random.seed(3141)
plr = DoubleMLPLR(dml_data,
                  ml_l=LGBMRegressor(**learner),
                  ml_m=LGBMRegressor(**learner),
                  score="partialling out", n_folds=3)
plr.fit(store_predictions=True)
print(f"average effect from Lab 4's estimator: {plr.coef[0]:+.3f}")

blp = plr.cate(basis, cov_type="cluster", cov_kwds={"groups": clusters})
models["II-3 PLR (boosted trees)"] = blp.blp_model

## The elasticity function

This corresponds to Table 8 of the paper. The `intercept` row is the average effect, and
every other row gives the shift in elasticity as that modifier increases.

In [ ]:
def summarise(res, level=CONFIDENCE):
    ci = res.conf_int(alpha=1 - level)
    return pd.DataFrame({
        "coef": res.params, "std err": res.bse, "t": res.tvalues,
        "P>|t|": res.pvalues, f"{(1-level)/2:.1%}": ci[0],
        f"{(1+level)/2:.1%}": ci[1],
    })


for name, res in models.items():
    print(f"\n=== {name} ===")
    print(summarise(res).round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
width = 0.26
x = np.arange(len(BASIS_COLS))

for i, (name, res) in enumerate(models.items()):
    ci = res.conf_int(alpha=1 - CONFIDENCE)
    coef = res.params.reindex(BASIS_COLS)
    lo, hi = ci[0].reindex(BASIS_COLS), ci[1].reindex(BASIS_COLS)
    ax.errorbar(x + (i - 1) * width, coef, yerr=[coef - lo, hi - coef],
                fmt="o", ms=6, capsize=4, capthick=1.5, lw=1.5,
                color=palette[i], ecolor=palette[i], label=name)

ax.axhline(0, color="red", ls="--", lw=1)
ax.set_xticks(x)
ax.set_xticklabels(BASIS_COLS, rotation=25, ha="right", fontsize=9)
ax.set_ylabel("coefficient")
ax.set_title(f"Estimated elasticity function, {CONFIDENCE:.0%} confidence intervals")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Are the modifiers jointly significant?

Individual coefficients on the similarities are noisy because a product's five cosine
similarities are nearly collinear: they are distances to five centroids on a sphere, so
knowing four almost determines the fifth. A joint test is the appropriate way to ask
whether product identity matters at all.

The same near-collinearity is why the test uses $k-1$ degrees of freedom for $k$
similarities, following the paper.

In [ ]:
def chi2_test(res, idx):
    """Wald test that the selected coefficients are jointly zero."""
    coef = res.params.iloc[idx].to_numpy()
    cov = res.cov_params().to_numpy()[np.ix_(idx, idx)]
    stat = coef @ np.linalg.solve(cov, coef)
    return 1 - chi2.cdf(stat, len(idx) - 1)


joint = pd.DataFrame({
    "All modifiers": {n: chi2_test(r, MOD_IDX) for n, r in models.items()},
    "Similarities only": {n: chi2_test(r, SIM_IDX) for n, r in models.items()},
})
print(f"chi-squared p-values (sample size: {len(data):,})")
joint.round(4)

## Group average effects

A coarser cut: assign each product to the cluster it is most similar to and estimate one
average effect per group.

In [ ]:
groups = pd.DataFrame(data[sim_cols].idxmax(axis=1), columns=["cluster"])
print(groups.value_counts().to_string())

gate = plr.gate(groups)
gate_summary = gate.confint(level=CONFIDENCE)

fig, ax = plt.subplots(figsize=(8, 4.5))
y = np.arange(len(gate_summary))
coef = gate_summary.iloc[:, 1]
ax.errorbar(coef, y,
            xerr=[coef - gate_summary.iloc[:, 0], gate_summary.iloc[:, 2] - coef],
            fmt="o", ms=7, capsize=4, capthick=1.5, lw=1.5, color=palette[0])
ax.axvline(plr.coef[0], color=palette[3], ls="--", lw=1.2, label="average effect")
ax.axvline(0, color="gray", lw=1)
ax.set_yticks(y)
ax.set_yticklabels([s.replace("similarity_", "") for s in gate_summary.index])
ax.set_xlabel("group average price effect")
ax.set_title(f"Effect by nearest cluster, {CONFIDENCE:.0%} CI")
ax.legend(fontsize=9)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

gate_summary.round(3)

## The distribution of elasticities

Predict $\hat\alpha(x)$ for every observation, sort ascending and plot. The spread of the
resulting curve is the heterogeneity.

Two bands are shown. The pointwise band covers the elasticity of a single product at a
given rank. The uniform band, obtained by a multiplier bootstrap of the maximal
$t$-statistic, covers the whole curve at once, and it is the one to use before claiming
that products at the left end are more elastic than those at the right.

In [ ]:
def effect_ci(res, basis, idx, level=CONFIDENCE, joint=False,
              n_boot=1000, seed=SEED):
    """Pointwise or uniform confidence band for basis @ coefficients."""
    b = basis.iloc[:, idx].to_numpy()
    coef = res.params.iloc[idx].to_numpy()
    cov = res.cov_params().to_numpy()[np.ix_(idx, idx)]

    effect = b @ coef
    se = np.sqrt(np.einsum("ij,jk,ik->i", b, cov, b))

    if joint:
        rng = np.random.default_rng(seed)
        root = np.real(sqrtm(cov))
        draws = (b @ root @ rng.normal(size=(len(idx), n_boot))).T / se
        crit = np.quantile(np.max(np.abs(draws), axis=0), level)
    else:
        crit = norm.ppf(0.5 + level / 2)

    return pd.DataFrame({"lower": effect - crit * se, "effect": effect,
                         "upper": effect + crit * se})


ALL_IDX = list(range(len(BASIS_COLS)))
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)

for ax, (name, res) in zip(axes, models.items()):
    point = effect_ci(res, basis, ALL_IDX, joint=False)
    unif = effect_ci(res, basis, ALL_IDX, joint=True)
    order = np.argsort(point["effect"].values)
    pct = np.linspace(0, 100, len(order))

    ax.fill_between(pct, unif["lower"].values[order], unif["upper"].values[order],
                    color=palette[0], alpha=0.15, label="uniform 90% band")
    ax.fill_between(pct, point["lower"].values[order], point["upper"].values[order],
                    color=palette[0], alpha=0.35, label="pointwise 90% band")
    ax.plot(pct, point["effect"].values[order], color=palette[0], lw=1.8)
    ax.axhline(res.params["intercept"], color=palette[3], ls="--", lw=1.2,
               label="average effect")
    ax.axhline(0, color="gray", lw=1)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("percentile of the estimated elasticity")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("$\\hat{\\alpha}(x)$  (rank elasticity)")
axes[0].legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
sorted_effects = effect_ci(models["II-3 PLR (boosted trees)"], basis, ALL_IDX)
rank = sorted_effects["effect"]

print("Estimated rank elasticity across products (Model II-3)")
print(rank.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(3).to_string())
print(f"\nimplied demand elasticity, 5th to 95th percentile: "
      f"{RANK_TO_DEMAND * rank.quantile(0.05):.2f} to "
      f"{RANK_TO_DEMAND * rank.quantile(0.95):.2f}")

## Where does the variation come from?

The estimated elasticity is a function of seven numbers. Regressing $\hat\alpha(x)$ on
each block in turn shows how the explanatory work divides between what a product is and
where it currently sits.

In [ ]:
alpha_hat = sorted_effects["effect"].values
blocks = {"256 embeddings": emb_cols,
          "5 similarities": sim_cols,
          "lagged Q, P": lag_controls}

share = {name: sm.OLS(alpha_hat, sm.add_constant(data[cols])).fit().rsquared
         for name, cols in blocks.items()}
print("share of Var(alpha-hat) explained")
for name, r2 in share.items():
    print(f"  {name:18s} {r2:6.1%}")

## What we found

**Price sensitivity is not constant.** The sorted-effects curve slopes across a wide
range. In the paper's full sample the rank elasticity runs from about $-2.0$ to $+0.4$,
implying demand elasticities from roughly $-4.0$ to $+0.8$ around an average near $-1.5$.
Pricing every product off the average would misprice both tails.

**Popularity is the strongest single modifier.** In the paper's full sample the
coefficient on lagged quantity is $-0.325$ with $t = -3.5$, so products that already sell
well lose proportionally more when their price rises. Lagged price enters negatively as
well, but is not significant even there. At this sample size expect the right sign and a
wide interval, since these are the coefficients a smaller sample affects most.

**Product identity matters as well.** Individual similarity coefficients are imprecise,
given the near-collinearity, but the joint $\chi^2$ test rejects, so the embeddings carry
information about which products are price-sensitive.

### The two roles of an embedding

Reading Labs 3, 4 and 5 together gives the paper's central claim.

| | Confounder? | Effect modifier? |
|---|---|---|
| **Evidence** | Lab 3: embeddings barely predict $\Delta P$ | Lab 5: joint $\chi^2$ rejects, wide spread of $\hat\alpha(x)$ |
| **Consequence** | Lab 4: adding them scarcely moves $\hat\delta$ | Lab 5: they identify which products are elastic |

The embeddings are weak confounders and strong effect modifiers. A homogeneous-effects
model therefore recovers a reasonable average while being unreliable for any particular
product. The value of representing products with AI here is not a better average estimate
but knowing which products the average fails to describe.

> **A note on your numbers.** Heterogeneity is what suffers most at this sample size.
> Estimating seven modifier coefficients on about 6,500 observations, against 38,041 in
> the paper, leaves the individual similarity coefficients wide, and the uniform band may
> cover the whole curve. Read the joint $\chi^2$ test and the spread of $\hat\alpha(x)$
> rather than individual $t$-statistics. Adding image part-files and raising
> `N_PRODUCTS` in Labs 1 and 2 sharpens the picture.